# ROC Curve & ROC-AUC (Area Under the Curve)

When evaluating classification models, metric scores like Accuracy, Precision, and Recall are evaluated at a **single specific threshold** (usually 0.5). To understand a model's performance holistically across all possible thresholds, we use the **ROC Curve** and the **ROC-AUC** metric.

In these notes, we will cover:
1. **The Need for ROC Curves:** Why evaluating thresholds matters, especially when dealing with imbalanced error costs.
2. **True Positive Rate (TPR) vs. False Positive Rate (FPR):** Formulating the benefit and cost of classifier decisions.
3. **The ROC Curve:** Plotting the trade-offs of TPR vs. FPR across all thresholds.
4. **ROC-AUC Score:** Using the area under the curve to summarize performance into a single metric and compare different machine learning models.
5. **Jupyter Code Implementation:** Generating curves and calculating AUC scores using Scikit-Learn.

## 1. Why do we need an ROC Curve?

### Threshold Selection
Most machine learning classification models (e.g. Logistic Regression) do not predict a class label directly. Instead, they output a **probability value** (between $0.0$ and $1.0$). 
- To convert this probability into a hard label (e.g., $0$ or $1$), we must apply a **Classification Threshold** ($th$):
  $$\text{Prediction} = \begin{cases} 
      1 & \text{if } P(y=1|X) \ge th \\
      0 & \text{if } P(y=1|X) < th 
  \end{cases}$$
- By default, libraries like Scikit-Learn use a threshold of **$0.5$**. However, this default is rarely optimal for real-world business or medical applications.

### Handling Imbalanced Error Costs
In practice, different types of errors carry different costs:
- **Spam Filtering:** Classifying a legitimate client email as spam (False Positive) is far more costly than letting a spam email slip into the inbox (False Negative). To avoid False Positives, we want a **higher threshold** (e.g., $0.8$ or $0.9$).
- **Medical Diagnostics:** Failing to detect an aggressive cancer (False Negative) is life-threatening, while a false alarm (False Positive) leads to further testing. To avoid False Negatives, we want a **lower threshold** (e.g., $0.1$ or $0.2$).

The **ROC Curve** allows us to visualize model behavior across **all possible thresholds** ($1.0$ down to $0.0$), helping us select the threshold that perfectly balances our specific cost-benefit trade-offs.

## 2. Core Foundations: TPR and FPR

To construct the ROC Curve, we plot two key metrics calculated from the Confusion Matrix:

### 1. True Positive Rate (TPR)
Also known as **Sensitivity** or **Recall**, TPR represents the **benefit** of the model. It measures the proportion of actual positive cases that the model correctly identified:

$$\text{TPR} = \frac{TP}{TP + FN}$$

We want this value to be as close to **$1.0$** as possible.

### 2. False Positive Rate (FPR)
FPR represents the **cost** (or false alarm rate) of the model. It measures the proportion of actual negative cases that the model incorrectly flagged as positive:

$$\text{FPR} = \frac{FP}{FP + TN}$$

Mathematically, it is related to **Specificity** (True Negative Rate) as:
$$\text{FPR} = 1 - \text{Specificity} = 1 - \frac{TN}{TN + FP}$$

We want this value to be as close to **$0.0$** as possible.

## 3. The ROC Curve Explained

The **ROC (Receiver Operating Characteristic) Curve** is a 2D graphical plot where:
- **Y-axis:** True Positive Rate (TPR)
- **X-axis:** False Positive Rate (FPR)

### Plotting Mechanics
To construct the curve:
1. We sort all predictions by their predicted probabilities.
2. We iterate through thresholds from $1.0$ down to $0.0$.
3. At each threshold, we convert probabilities to predictions, compute the confusion matrix, calculate $(FPR, TPR)$, and plot that coordinate.
4. Connecting these points yields the ROC Curve.

### Understanding the Trade-off
- At a threshold of **$1.0$**: The model predicts $0$ for everything. Thus, $TP=0$ and $FP=0$, starting the curve at the origin **$(0, 0)$**.
- At a threshold of **$0.0$**: The model predicts $1$ for everything. Thus, $FN=0$ and $TN=0$. TPR becomes $1.0$ and FPR becomes $1.0$, ending the curve at **$(1, 1)$**.
- As we lower the threshold, we classify more points as positive. This increases our **benefit** (TPR goes up) but also increases our **cost** (FPR goes up).
- A perfect classifier would immediately jump from $(0,0)$ to $(0,1)$ and then to $(1,1)$, hugging the top-left corner.

## 4. ROC-AUC (Area Under the Curve)

Instead of comparing 2D curves visually, we can summarize the classifier's performance into a single scalar value: the **ROC-AUC Score**.

### Definition & Interpretation
ROC-AUC measures the two-dimensional area underneath the entire ROC Curve.
- **$\text{AUC} = 1.0$ (Perfect Classifier):** The model separates classes perfectly. There is a threshold where TPR is $1.0$ and FPR is $0.0$.
- **$\text{AUC} = 0.5$ (Random Guessing):** The model has no discriminatory power. It is represented by the diagonal line connecting $(0, 0)$ and $(1, 1)$.
- **$\text{AUC} < 0.5$ (Inverted Predictions):** The model is performing worse than random guessing. If you flip the model's predictions, it becomes a good classifier!

### Practical Use Case: Model Comparison
ROC-AUC is highly robust because it evaluates the model's performance **independent of the classification threshold**. This makes it the standard metric for comparing different algorithms (e.g., comparing Logistic Regression vs. Naive Bayes or Random Forest) to determine which model is a superior classifier overall.

## 5. Setup and Libraries Import

Let's set up our workspace and import the required tools.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_curve, roc_auc_score

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

print("Libraries imported successfully!")

## 6. Dataset Generation & Model Training

We will generate a synthetic classification dataset and train two different models:
1. **Logistic Regression** (A strong linear classifier).
2. **Naive Bayes** (A baseline probabilistic classifier).

In [ ]:
# Generate synthetic dataset
X, y = make_classification(
    n_samples=1000, 
    n_features=10, 
    n_informative=5,
    n_classes=2, 
    random_state=42
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train models
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict probabilities (Crucial for ROC)
# We select the probabilities of the positive class (column index 1)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
nb_probs = nb_model.predict_proba(X_test)[:, 1]

print("Models trained and probabilities extracted!")

## 7. Computing and Plotting the ROC Curves

We will compute the TPR, FPR, and thresholds for both models using `sklearn.metrics.roc_curve`, calculate the AUC scores using `sklearn.metrics.roc_auc_score`, and plot them.

In [ ]:
# Calculate TPR and FPR for various thresholds
lr_fpr, lr_tpr, lr_thresholds = roc_curve(y_test, lr_probs)
nb_fpr, nb_tpr, nb_thresholds = roc_curve(y_test, nb_probs)

# Calculate AUC scores
lr_auc = roc_auc_score(y_test, lr_probs)
nb_auc = roc_auc_score(y_test, nb_probs)

# Plot ROC Curves
plt.figure(figsize=(10, 8))

# Plot the diagonal line representing random guessing (AUC = 0.5)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=2, label='Random Guessing (AUC = 0.50)')

# Plot Logistic Regression curve
plt.plot(lr_fpr, lr_tpr, color='#9b59b6', linewidth=3, label=f'Logistic Regression (AUC = {lr_auc:.4f})')

# Plot Naive Bayes curve
plt.plot(nb_fpr, nb_tpr, color='#3498db', linewidth=3, label=f'Naive Bayes (AUC = {nb_auc:.4f})')

# Highlight threshold locations on the Logistic Regression curve (e.g. 0.5, 0.8, 0.2)
target_thresholds = [0.2, 0.5, 0.8]
for th in target_thresholds:
    # Find the index of the closest threshold
    idx = np.argmin(np.abs(lr_thresholds - th))
    plt.scatter(lr_fpr[idx], lr_tpr[idx], color='#e74c3c', edgecolor='black', s=120, zorder=5)
    plt.annotate(
        f'th={th:.1f}', 
        xy=(lr_fpr[idx], lr_tpr[idx]), 
        xytext=(15, -10), 
        textcoords='offset points', 
        arrowprops=dict(arrowstyle="->", color='black'),
        fontsize=11,
        fontweight='bold'
    )

plt.title('ROC Curve Comparison: Logistic Regression vs. Naive Bayes', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.legend(frameon=True, facecolor='white', fontsize=11, loc='lower right')
plt.tight_layout()
plt.show()

print(f"Logistic Regression AUC: {lr_auc:.6f}")
print(f"Naive Bayes AUC:         {nb_auc:.6f}")

## Summary Checklist for ROC & ROC-AUC

1. **Evaluate Across Thresholds:** Use the **ROC Curve** to understand model trade-offs across all classification thresholds rather than evaluating only the default 0.5.
2. **True Positive Rate (TPR):** Measures sensitivity (maximize benefit).
3. **False Positive Rate (FPR):** Measures false alarms (minimize cost).
4. **Use ROC-AUC for Comparison:** The Area Under the Curve (AUC) is threshold-independent, providing an excellent single-metric benchmark for comparing different models (AUC = 1.0 is perfect, AUC = 0.5 is random guessing).
5. **Threshold Tuning:** Once a model is selected, use the plotted curve coordinates to select a custom threshold matching the business cost profile.